# Dataset Splits for Training Data Preparation

[![Open In Colab](https://img.shields.io/badge/Open%20In-Colab-blue?style=for-the-badge&logo=google-colab)](https://colab.research.google.com/github/dnth/rag-datakit/blob/main/nbs/01_dataset-splits.ipynb)
[![Open In Kaggle](https://img.shields.io/badge/Open%20In-Kaggle-blue?style=for-the-badge&logo=kaggle)](https://kaggle.com/kernels/welcome?src=https://github.com/dnth/rag-datakit/blob/main/nbs/01_dataset-splits.ipynb)

This notebook prepares synthetic training data for embedding model training by creating proper train/validation splits. We'll work with the synthetic triplets generated from the Singapore SkillsFuture Framework dataset to create training and validation sets suitable for embedding model fine-tuning.

## What you'll learn:
- How to load synthetic datasets from Hugging Face Hub
- Creating proper train/validation splits for embedding training
- Data preprocessing for triplet-based training
- Publishing split datasets back to Hugging Face Hub

## Installation

Install the rag-datakit package which includes all necessary dependencies including distilabel, transformers, and dataset utilities. Uncomment the cell below to install if you haven't already.

In [ ]:
# !pip install git+https://github.com/dnth/rag-datakit.git

## Dataset Loading and Preprocessing

Load the synthetic dataset generated from the previous notebook. We'll work with the triplet data (anchor, positive, negative) that was created using distilabel for embedding training purposes.

**Dataset source**: Synthetic job description triplets from Singapore Skills Framework  
**HuggingFace repo**: `dnth/ssf-synthetic-data-for-retriever-openai`

The dataset contains triplets for embedding training:
- **anchor**: Original job role descriptions
- **positive**: Paraphrased versions (semantically similar)
- **negative**: Different job descriptions (semantically dissimilar)

In [1]:
from datasets import load_dataset, concatenate_datasets

dataset_easy = load_dataset("dnth/ssf-dataset-synthetic-v4.2", "easy_triplets")
dataset_easy_v2 = load_dataset("dnth/ssf-dataset-synthetic-v4.2", "easy_triplets_v2")
dataset_easy_v3 = load_dataset("dnth/ssf-dataset-synthetic-v4.2", "easy_triplets_v3")

dataset_hard = load_dataset("dnth/ssf-dataset-synthetic-v4.2", "hard_triplets")
dataset_hard_v2 = load_dataset("dnth/ssf-dataset-synthetic-v4.2", "hard_triplets_v2")


README.md: 0.00B [00:00, ?B/s]

easy_triplets/train-00000-of-00001.parqu(…):   0%|          | 0.00/4.73M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1885 [00:00<?, ? examples/s]

easy_triplets_v2/train-00000-of-00001.pa(…):   0%|          | 0.00/4.73M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1885 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/1885 [00:00<?, ? examples/s]

hard_triplets/train-00000-of-00001.parqu(…):   0%|          | 0.00/4.93M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1885 [00:00<?, ? examples/s]

hard_triplets_v2/train-00000-of-00001.pa(…):   0%|          | 0.00/4.92M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1885 [00:00<?, ? examples/s]

In [2]:
dataset = concatenate_datasets(
    [
        dataset_easy["train"],
        dataset_easy_v2["train"],
        dataset_easy_v3["train"],
        dataset_hard["train"],
        dataset_hard_v2["train"],
    ]
)
dataset

Dataset({
    features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
    num_rows: 9425
})

In [3]:
# Convert Sector to ClassLabel
from datasets import ClassLabel

dataset = dataset.class_encode_column("Sector")
dataset

Casting to class labels:   0%|          | 0/9425 [00:00<?, ? examples/s]

Dataset({
    features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
    num_rows: 9425
})

In [4]:
dataset = dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="Sector")

In [5]:
dataset = dataset.select_columns(["Sector","anchor", "positive", "negative"])
dataset

DatasetDict({
    train: Dataset({
        features: ['Sector', 'anchor', 'positive', 'negative'],
        num_rows: 7540
    })
    test: Dataset({
        features: ['Sector', 'anchor', 'positive', 'negative'],
        num_rows: 1885
    })
})

## Publishing Final Dataset

Upload the properly split dataset to Hugging Face Hub for easy access during embedding model training. The dataset will be structured as a `DatasetDict` with separate train and validation splits.

This makes the dataset readily available for:
- Embedding model fine-tuning scripts
- Reproducible training experiments  
- Sharing with team members or the community
- Integration with training frameworks like sentence-transformers

In [6]:
from datasets import DatasetDict

ds = DatasetDict({"train": dataset['train'], "valid": dataset['test']})

ds

DatasetDict({
    train: Dataset({
        features: ['Sector', 'anchor', 'positive', 'negative'],
        num_rows: 7540
    })
    valid: Dataset({
        features: ['Sector', 'anchor', 'positive', 'negative'],
        num_rows: 1885
    })
})

In [7]:
ds.push_to_hub("dnth/ssf-train-valid-v4.2")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   6%|6         |  526kB / 8.51MB            

...

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  24%|##4       |  525kB / 2.16MB            

CommitInfo(commit_url='https://huggingface.co/datasets/dnth/ssf-train-valid-v4.2/commit/97c8b4d3dc96a480e369838fb9f00464ce9080e9', commit_message='Upload dataset', commit_description='', oid='97c8b4d3dc96a480e369838fb9f00464ce9080e9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/dnth/ssf-train-valid-v4.2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='dnth/ssf-train-valid-v4.2'), pr_revision=None, pr_num=None)